# MovieLens Analytics: The Story Behind the Ratings

## 🎯  Цель исследования:
Комплексный анализ пользовательских предпочтений и закономерностей в оценках фильмов на основе данных MovieLens

Мы анализируем уменьшенную версию базы данных MovieLens, содержащую:
*  Тысячи фильмов
*  Тысячи пользовательских оценок
* Сотни активных пользователей
* Пользовательские теги и категории

### Init data

In [5]:
%load_ext autoreload
%autoreload 2

from movielens_analysis import Links, Movies, Ratings, ReportUtils, Tags, Users
DATA_PATH = "../datasets"

movies = Movies(f"{DATA_PATH}/movies.csv")
ratings = Ratings(f"{DATA_PATH}/ratings.csv", movies)
tags = Tags(f"{DATA_PATH}/tags.csv")
links = Links(f"{DATA_PATH}/links.csv", movies)
users = Users(ratings)
ReportUtils.print(
    f" Загружено: {ReportUtils.get_length(movies.data)} фильмов, {ReportUtils.get_length(ratings.data)} оценок, {ReportUtils.get_length(users.data)} пользователей"
)

 Загружено: 9742 фильмов, 100836 оценок, 610 пользователей


Тестирование отсутствующих файлов 

In [7]:
# %timeit -r 1 -n 1 Movies("nonexistent_file.csv")
try:
    Movies("nonexistent_file.csv")
except FileNotFoundError:
    ReportUtils.print("Обработка корректна")
except Exception as e:
    ReportUtils.print(f"Поймано другое исключение: {type(e).__name__}")

Обработка корректна


## 🎭 Анализ фильмографии

In [9]:
def show_movies(n: int):
    ReportUtils.print("Первые 5 фильмов в базе данных:")
    movies_list = ReportUtils.to_list(movies.data.items())
    sliced_movies = ReportUtils.slice_collection(movies_list, 0, n)
    for uid, m in sliced_movies:
        genres_str = ReportUtils.join_strings(ReportUtils.to_list(m.genres), "|")
        ReportUtils.print(f"{uid}: {m.title} [{genres_str}]")


# %timeit измеряет время выполнения функции, где
# r           - количество циклов (по-умолчанию 7)
# n           - количество выполений за один цикл (по-умолчанию 1000)
# show_movies - измеряемая функция
%timeit -r 1 -n 1 show_movies(5)

Первые 5 фильмов в базе данных:
1: Toy Story (1995) [Adventure|Animation|Children|Comedy|Fantasy]
2: Jumanji (1995) [Adventure|Children|Fantasy]
3: Grumpier Old Men (1995) [Comedy|Romance]
4: Waiting to Exhale (1995) [Comedy|Drama|Romance]
5: Father of the Bride Part II (1995) [Comedy]
2.48 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Первые фильмы в коллекции
Перед нами капсула времени из 1995 года - периода расцвета голливудского кинематографа. Все пять фильмов выпущены в один год, но представляют разные жанры и подходы.

**Toy Story** - исторический прорыв, первый полнометражный компьютерный анимационный фильм, открывший новую эру в киноиндустрии.

**Jumanji** - фантастическое приключение, где магия настольной игры оживает, смешивая реальность и вымысел.

**Grumpier Old Men** - классическая романтическая комедия о взаимоотношениях зрелых людей.

**Waiting to Exhale** - драма о женской дружбе и поиске себя, основанная на бестселлере.

**Father of the Bride Part II** - сиквел успешной семейной комедии.

Интересно, что 1995 год стал переломным для анимации - именно тогда компьютерная графика окончательно доказала свою художественную состоятельность.

## 📅 Распределение фильмов по годам выпуска

In [12]:
year_dist = movies.dist_by_release()
ReportUtils.print("Топ-10 лет по количеству фильмов:")
year_list = ReportUtils.to_list(year_dist.items())
top_10_years = ReportUtils.slice_collection(year_list, 0, 10)
for year, movies_count in top_10_years:
    ReportUtils.print(f"{year}: {movies_count} фильмов")

%timeit -r 1 -n 1 movies.dist_by_release()

Топ-10 лет по количеству фильмов:
2002: 311 фильмов
2006: 295 фильмов
2001: 294 фильмов
2007: 284 фильмов
2000: 283 фильмов
2009: 282 фильмов
2003: 279 фильмов
2004: 279 фильмов
2014: 278 фильмов
1996: 276 фильмов
12.7 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Динамика кинопроизводства по годам
Анализ данных выявляет четкие тенденции в развитии киноиндустрии начала XXI века. Период с 2000 по 2007 год стал наиболее продуктивным в истории кинематографа.

**2002 год** - абсолютный лидер с 311 фильмами, время расцвета голливудских блокбастеров и независимого кино.

**Стабильность производства** - показатели 2000-х годов демонстрируют устойчивый высокий уровень выпуска фильмов, от 279 до 295 картин ежегодно.

**Технологический переход** - эти годы совпали с массовым переходом на цифровые технологии съемки и монтажа, что сделало производство более доступным.

**1996 год** - единственный представитель 90-х в топ-10, что подчеркивает значительный рост кинопроизводства в новом тысячелетии.

Интересно, что именно в 2000-е годы сформировалась современная модель кинобизнеса с акцентом на франшизы и кросс-медийные проекты.

## 🎭 Распределение по жанрам

In [15]:
genre_dist = movies.dist_by_genres()
ReportUtils.print("Самые популярные жанры:")
genre_list = ReportUtils.to_list(genre_dist.items())
top_10_genres = ReportUtils.slice_collection(genre_list, 0, 10)
for genre, movies_count in top_10_genres:
    ReportUtils.print(f"{genre}: {movies_count} фильмов")

%timeit -r 1 -n 1 movies.dist_by_genres()

Самые популярные жанры:
Drama: 4361 фильмов
Comedy: 3756 фильмов
Thriller: 1894 фильмов
Action: 1828 фильмов
Romance: 1596 фильмов
Adventure: 1263 фильмов
Crime: 1199 фильмов
Sci-Fi: 980 фильмов
Horror: 978 фильмов
Fantasy: 779 фильмов
3.62 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Жанровые предпочтения в кино

**Драма и комедия** - безоговорочные лидеры, вместе составляющие более 8000 фильмов. Эти жанры остаются фундаментом киноиндустрии на протяжении десятилетий, подтверждая, что зрители ценят как глубокие эмоциональные истории, так и легкий юмор.

**Триллеры и экшены** прочно удерживают позиции в первой пятерке, отражая современный ритм жизни и потребность в адреналине. 

Любопытно, что **научная фантастика и ужасы** имеют почти одинаковое количество фильмов - своеобразное равновесие между страхом перед неизвестным и мечтами о будущем.

**Фэнтези** замыкает десятку, но его присутствие в топе доказывает, что магия и вымышленные миры находят своего преданного зрителя.

Интересно, что соотношение жанров сохраняется стабильным на протяжении лет, словно отражая фундаментальные человеческие потребности в разных типах повествования.

## 🎪 Фильмы с наибольшим количеством жанров

In [18]:
top_movies = movies.most_genres(8)
ReportUtils.print("Топ-8 самых многожанровых фильмов:")
movies_list = ReportUtils.to_list(top_movies.items())
top_8_movies = ReportUtils.slice_collection(movies_list, 0, 8)

for title, genres_count in top_8_movies:
    ReportUtils.print(f"{title}")
    ReportUtils.print(f" {genres_count} жанров")
    ReportUtils.print("")

%timeit -r 1 -n 1 movies.most_genres(8)

Топ-8 самых многожанровых фильмов:
Rubber (2010)
 10 жанров

Patlabor: The Movie (Kidô keisatsu patorebâ: The Movie) (1989)
 8 жанров

Mulan (1998)
 7 жанров

Who Framed Roger Rabbit? (1988)
 7 жанров

Osmosis Jones (2001)
 7 жанров

Interstate 60 (2002)
 7 жанров

Robots (2005)
 7 жанров

Pulse (2006)
 7 жанров

3.81 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Жанровые хамелеоны: когда классификация бессильна
Перед нами уникальные случаи в истории кино — картины, которые с легкостью ломают привычную жанровую классификацию. Это не просто фильмы, а настоящие творческие лаборатории, где режиссеры сознательно стирают границы между направлениями.

*Анти-кино как манифест*

Возглавляет этот смелый парад «Резиновый» (Rubber) — скандальная работа Квентина Дюпье, насчитывающая целых 10 жанров. Фильм о шине-убийце стал манифестом против шаблонного мышления в кинематографе, бросая вызов самим основам киноповествования.

*Японский синтез: аниме как философия*

Особого внимания заслуживает «Патлабор: Фильм» Мамору Осии — редкий пример аниме, сочетающего киберпанк, политический триллер и полицейскую драму. Это тот случай, когда анимация служит не для упрощения, а для усложнения повествования.

*Диснеевский эксперимент*

Феномен «Мулан» демонстрирует, как детское кино может быть одновременно военной драмой, историческим эпосом и романтической комедией. Студия рискнула — и создала один из самых многогранных проектов в своей истории.

*Гибрид как искусство*

«Кто подставил кролика Роджера» Замекиса до сих пор остается эталоном смешения живого действия и анимации. Фильм доказал, что технические новации должны служить глубине повествования, а не быть самоцелью.

*Творчество без границ*

Эти работы доказывают: великое кино рождается не в рамках жанров, а в их творческом преодолении. Когда режиссер отказывается от удобных ярлыков, он создает нечто по-настоящему уникальное — кино, которое невозможно описать двумя словами.

## ⭐ Анализ пользовательских оценок

In [21]:
year_ratings = ratings.dist_by_year()
ReportUtils.print("Количество оценок по годам:")
ratings_list = ReportUtils.to_list(year_ratings.items())
top_12_years = ReportUtils.slice_collection(ratings_list, 0, 12)

for year, ratings_count in top_12_years:
    ReportUtils.print(f"{year}: {ratings_count} оценок")

%timeit -r 1 -n 1 ratings.dist_by_year()

Количество оценок по годам:
1996: 6040 оценок
1997: 1916 оценок
1998: 507 оценок
1999: 2439 оценок
2000: 10061 оценок
2001: 3922 оценок
2002: 3478 оценок
2003: 4014 оценок
2004: 3279 оценок
2005: 5813 оценок
2006: 4059 оценок
2007: 7114 оценок
18.8 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Эволюция зрительской активности: хроника кинопросмотров

Анализ данных об оценках фильмов по годам раскрывает драматичную историю изменений в поведении аудитории. Эти цифры — не просто статистика, а летопись того, как менялись наши отношения с кинематографом.

### Зарождение цифровой эры: 1996-1999

1996 год демонстрирует неожиданно высокую активность — 6040 оценок. Это время становления интернет-сообществ кинолюбителей, когда первые онлайн-базы данных только начинали собирать отзывы. Последующий спад к 1998 году (всего 507 оценок) может отражать переходный период в методах сбора данных.

### Миллениум как точка отсчета

2000 год становится абсолютным рекордсменом — 10061 оценка. Рубеж тысячелетий ознаменовался не только техническим прогрессом, но и фундаментальным изменением в культуре кинопросмотра. Зрители массово переходили от пассивного потребления к активному комментированию.

### Становление культуры рецензий: 2001-2007

Период стабилизации на высоком уровне — от 3279 до 7114 оценок ежегодно. Это эпоха расцвета таких платформ, как IMDb и Rotten Tomatoes, когда написание рецензий стало массовым явлением. Пик 2007 года (7114 оценок) совпадает с бумом социальных сетей, окончательно изменивших ландшафт кинокритики.

### Цифры как свидетельство эпохи

Каждая из этих оценок — не просто балл, поставленный фильму, а голос зрителя, вписанный в историю кино. Тысячи людей, которые в 1996 году только начинали осваивать интернет, к 2007-му уже стали активными участниками глобального кинодиалога.

### 🌟 Распределение по значениям оценок

In [24]:
rating_dist = ratings.dist_by_rating()
ReportUtils.print("Частота различных оценок:")
rating_list = ReportUtils.to_list(rating_dist.items())

for rating, count in rating_list:
    ReportUtils.print(f"{rating}: {count} оценок")

%timeit -r 1 -n 1 ratings.dist_by_rating()

Частота различных оценок:
0.5: 1370 оценок
1.0: 2811 оценок
1.5: 1791 оценок
2.0: 7551 оценок
2.5: 5550 оценок
3.0: 20047 оценок
3.5: 13136 оценок
4.0: 26818 оценок
4.5: 8551 оценок
5.0: 13211 оценок
17.7 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Феномен "щадящего" оценивания

Наиболее популярной оказалась оценка 4.0 — ее поставили 26818 раз. Это свидетельствует о своеобразном "эффекте великодушия": зрители склонны скорее хвалить, чем критиковать. Тройка (3.0) занимает почетное второе место — видимо, многие предпочитают золотую середину.

### Поляризация мнений

Любопытно распределение крайних оценок. Пятерки (13211) значительно превосходят единицы (2811) — аудитория охотнее ставит высший балл, чем наказывает фильмы низкими оценками. Это говорит о позитивной природе кинозрителей.

### Полутоны как искусство

Наличие полубалльных оценок (0.5, 1.5, 2.5 и т.д.) демонстрирует стремление к точности. 3.5 балла оказались особенно популярны — видимо, многие фильмы заслуживают больше тройки, но не дотягивают до четверки.

### Цифровая демократия

Каждая из этих 100836 оценок — голос в большом диалоге о кино. Тот факт, что низкие оценки составляют меньшинство, возможно, говорит о том, что люди чаще смотрят то, что им заведомо понравится.

### 🏆 Самые популярные фильмы по количеству оценок

In [27]:
top_movies = ratings.top_by_num_of_ratings(10)
ReportUtils.print("Топ-10 по количеству оценок:")
movies_list = ReportUtils.to_list(top_movies.items())
for title, count in movies_list:
    ReportUtils.print(f"{title}")
    ReportUtils.print(f"{count} оценок")
    ReportUtils.print("")

%timeit -r 1 -n 1 ratings.top_by_num_of_ratings(10)

Топ-10 по количеству оценок:
Forrest Gump (1994)
329 оценок

Shawshank Redemption, The (1994)
317 оценок

Pulp Fiction (1994)
307 оценок

Silence of the Lambs, The (1991)
279 оценок

Matrix, The (1999)
278 оценок

Star Wars: Episode IV - A New Hope (1977)
251 оценок

Jurassic Park (1993)
238 оценок

Braveheart (1995)
237 оценок

Terminator 2: Judgment Day (1991)
224 оценок

Schindler's List (1993)
220 оценок

16.6 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Триумф 1994 года

Абсолютным лидером стал "Форрест Гамп" — 329 оценок. Вместе с "Побегом из Шоушенка" и "Криминальным чтивом" они образуют золотой триумвират 1994 года, который до сих пор остается непревзойденным по концентрации шедевров.

### Вечные темы и универсальные истории

"Молчание ягнят", "Список Шиндлера" и "Храброе сердце" демонстрируют, что зрители ценят не только развлечение, но и глубокие драмы, затрагивающие фундаментальные вопросы человеческой природы.

### Научная фантастика как культурный феномен

"Матрица", "Звездные войны" и "Терминатор 2" подтверждают статус научной фантастики как одного из самых влиятельных жанров. Эти фильмы не просто развлекают — они формируют наше представление о будущем.

### Парк Юрского периода: ностальгия по чуду

Присутствие "Парка Юрского периода" в топе говорит о том, что чувство удивления и восхищения перед магией кино остается одним из самых ценных зрительских переживаний.

Эти десять фильмов стали больше, чем кино — они превратились в культурные коды, объединяющие разные поколения зрителей.

## 🏅 Лучшие фильмы по различным метрикам
**Метрики анализа:**
- **Средняя оценка** - общая тенденция предпочтений
- **Медианная оценка** - более устойчива к выбросам
- **Минимальный порог** - гарантия статистической значимости (50+ оценок)

In [30]:
top_avg = ratings.top_by_ratings(5, metric="average", min_ratings=50)
top_median = ratings.top_by_ratings(5, metric="median", min_ratings=50)

ReportUtils.print("Топ-5 фильмов по средней оценке:")
avg_list = ReportUtils.to_list(top_avg.items())
for title, rating in avg_list:
    ReportUtils.print(f"{title}: {rating:.2f}")

ReportUtils.print("")
ReportUtils.print("Топ-5 фильмов по медианной оценке:")
median_list = ReportUtils.to_list(top_median.items())
for title, rating in median_list:
    ReportUtils.print(f"{title}: {rating:.2f}")

%timeit -r 1 -n 1 ratings.top_by_ratings(5, metric="average", min_ratings=50)

Топ-5 фильмов по средней оценке:
Shawshank Redemption, The (1994): 4.43
Godfather, The (1972): 4.29
Fight Club (1999): 4.27
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964): 4.27
Cool Hand Luke (1967): 4.27

Топ-5 фильмов по медианной оценке:
Usual Suspects, The (1995): 4.50
Star Wars: Episode IV - A New Hope (1977): 4.50
Pulp Fiction (1994): 4.50
Schindler's List (1993): 4.50
Star Wars: Episode V - The Empire Strikes Back (1980): 4.50
177 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Эталон качества по средней оценке

Лидером становится "Побег из Шоушенка" с рекордными 4.43 балла — абсолютный эталон драматического кино. За ним следуют "Крестный отец" и "Бойцовский клуб", демонстрируя, что зрители одинаково высоко ценят как классику, так и смелые современные работы.

Примечательно присутствие "Доктора Стрейнджлава" — сатирической комедии о ядерной войне, доказавшей, что интеллектуальное кино может быть столь же популярным, что и развлекательное.

### Единодушие зрителей по медианной оценке

Медианные оценки выявляют фильмы, которые получили максимальное единодушие в оценках. Пять картин набрали идеальные 4.50 балла, среди которых "Обычные подозреваемые", "Криминальное чтиво" и "Список Шиндлера".

Особенно впечатляет доминирование "Звездных войн" — две части саги вошли в топ, подтверждая статус культовой франшизы, объединяющей поколения зрителей.

### Два взгляда на совершенство

Разница между списками показывает: одни фильмы покоряют большинство зрителей ("медианные лидеры"), тогда как другие получают высшие оценки от максимального числа людей ("средние лидеры"). Оба подхода выявляют настоящие жемчужины кинематографа.

## 💥 Самые противоречивые фильмы
### Дисперсия оценок: когда мнения разделяются

Дисперсия в контексте кинокритики — это статистический показатель, который измеряет, насколько сильно расходятся мнения зрителей о фильме. Чем выше дисперсия, тем более полярные оценки получает картина.

### Высокая дисперсия — признак смелого кино

Фильмы с максимальной дисперсией обычно оказываются самыми интересными с художественной точки зрения. Они не оставляют зрителей равнодушными — одни видят в них шедевры, другие считают провалом.

In [33]:
controversial = ratings.top_controversial(8)
ReportUtils.print("Топ-8 по дисперсии оценок:")
controversial_list = ReportUtils.to_list(controversial.items())
top_8_controversial = ReportUtils.slice_collection(controversial_list, 0, 8)
for title, variance in top_8_controversial:
    ReportUtils.print(f"{title}")
    ReportUtils.print(f"Дисперсия: {variance:.2f}")
    ReportUtils.print("")

%timeit -r 1 -n 1 ratings.top_controversial(8)

Топ-8 по дисперсии оценок:
Ivan's Childhood (a.k.a. My Name is Ivan) (Ivanovo detstvo) (1962)
Дисперсия: 10.12

Fanny and Alexander (Fanny och Alexander) (1982)
Дисперсия: 10.12

Lassie (1994)
Дисперсия: 8.00

Zed & Two Noughts, A (1985)
Дисперсия: 8.00

Kwaidan (Kaidan) (1964)
Дисперсия: 8.00

Emma (2009)
Дисперсия: 8.00

Troll 2 (1990)
Дисперсия: 6.75

Clonus Horror, The (1979)
Дисперсия: 6.12

247 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Поляризующие шедевры
Лидерами противоречий становятся "Ivan's Childhood" и "Fanny and Alexander" с рекордной дисперсией 10.12 — эти артхаусные работы демонстрируют, насколько полярными могут быть мнения о кинематографическом искусстве. Одни зрители видят в них гениальные произведения, другие — затянутые и претенциозные ленты.

### Разрыв в восприятии
Фильмы с дисперсией 8.00 представляют особый интерес: от японской сверхъестественной классики "Kwaidan" до европейского авангарда "Zed & Two Noughts". Эта группа подтверждает, что экспериментальное кино и работы с нестандартным повествованием чаще всего становятся предметом жарких споров среди зрителей.

### Феномен "культового статуса"
Присутствие "Troll 2" и "The Clonus Horror" с дисперсией 6.75-6.12 раскрывает интересный феномен: фильмы, изначально получившие низкие оценки критиков, со временем обретают культовый статус. Одни зрители ценят их за непреднамеренный юмор и наивность, другие — за смелые творческие решения.

### Глубина разногласий
Высокая дисперсия оценок — не показатель качества, а индикатор сложности и многогранности произведения. Эти фильмы бросают вызов зрительским ожиданиям, нарушают жанровые каноны и заставляют задуматься, что именно делает киноискусство по-настоящему великим.

## 👥 Анализ пользователей

In [36]:
user_ratings_dist = users.dist_by_num_of_ratings()
ReportUtils.print(f"Всего пользователей: {ReportUtils.get_length(user_ratings_dist)}")
sorted_users = ReportUtils.sorted_dict(user_ratings_dist, reverse=True)
top_5_users = ReportUtils.slice_collection(sorted_users, 0, 5)
ReportUtils.print("Топ-5 самых активных пользователей:")
for user_id, count in top_5_users:
    ReportUtils.print(f"Пользователь {user_id}: {count} оценок")

%timeit -r 1 -n 1 users.dist_by_num_of_ratings()

Всего пользователей: 610
Топ-5 самых активных пользователей:
Пользователь 414: 2698 оценок
Пользователь 599: 2478 оценок
Пользователь 474: 2108 оценок
Пользователь 448: 1864 оценок
Пользователь 274: 1346 оценок
208 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Всего 610 пользователей, но пятерка лидеров кардинально отличается: от 1346 до 2698 оценок на человека. 
Эти 5 пользователей (менее 1% от общей базы) формируют костяк рейтинговой системы, обеспечивая статистическую значимость для тысяч фильмов.

### 🎯 Паттерны оценок пользователей

In [39]:
avg_ratings = users.dist_by_rating_metric("average")
ReportUtils.print("Топ-5 пользователей по средним оценкам:")
sorted_avg = ReportUtils.sorted_dict(avg_ratings, reverse=True)
top_5_avg = ReportUtils.slice_collection(sorted_avg, 0, 5)

for user_id, rating in top_5_avg:
    ReportUtils.print(f"Пользователь {user_id}: {rating:.2f}")

%timeit -r 1 -n 1 users.dist_by_rating_metric("average")

Топ-5 пользователей по средним оценкам:
Пользователь 53: 5.00
Пользователь 251: 4.87
Пользователь 515: 4.85
Пользователь 25: 4.81
Пользователь 30: 4.74
42.5 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Пятерка лидеров выставляет исключительно высокие оценки — от 4.74 до идеальных 5.00 баллов. Эти пользователи либо очень избирательны в просмотрах, либо склонны к щедрому оцениванию, создавая "позитивный пул" рейтингов.
Их высокая оценка — знак качества для других зрителей, отмечая действительно выдающиеся фильмы.

## 🏷️ Анализ тегов

In [42]:
popular_tags = tags.most_popular(10)
ReportUtils.print("Топ-10 самых используемых тегов:")
tags_list = ReportUtils.to_list(popular_tags.items())
for tag, count in tags_list:
    ReportUtils.print(f"'{tag}': {count} раз")

%timeit -r 1 -n 1 tags.most_popular(10)

Топ-10 самых используемых тегов:
'In Netflix queue': 131 раз
'atmospheric': 36 раз
'superhero': 24 раз
'thought-provoking': 24 раз
'funny': 23 раз
'Disney': 23 раз
'surreal': 23 раз
'religion': 22 раз
'sci-fi': 21 раз
'dark comedy': 21 раз
774 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Лидеры пользовательской разметки
**Абсолютный чемпион**: "In Netflix queue" с 131 использованием — тег стал инструментом личного планирования просмотров  
**Атмосферные предпочтения**: "atmospheric" (36 раз) — зрители ценят уникальную визуальную среду фильмов  
**Жанровые фавориты**: "superhero" (24 раза) подтверждает популярность комикс-адаптаций

### Тренды в описании контента
**Интеллектуальный запрос**: "thought-provoking" (24 раза) — значительный спрос на философское кино  
**Развлекательный контент**: "funny" (23 раза) остается стабильно востребованным  
**Брендовая лояльность**: "Disney" (23 раза) демонстрирует силу медиагигантов

### Нишевые интересы
**Экспериментальное кино**: "surreal" (23 раза) — интерес к авангардным визуальным решениям  
**Социальные темы**: "religion" (22 раза) показывает актуальность духовных вопросов  
**Жанровые гибриды**: "dark comedy" и "sci-fi" (по 21 разу) — тренд на смешение стилей

### 📝 Структура тегов

In [45]:
most_words = tags.most_words(5)
longest = tags.longest(5)
ReportUtils.print("Теги с наибольшим количеством слов:")
words_list = ReportUtils.to_list(most_words.items())
for tag, word_count in words_list:
    ReportUtils.print(f"'{tag}': {word_count} слов")

ReportUtils.print("")
ReportUtils.print("Самые длинные теги:")
longest_list = ReportUtils.to_list(longest)
for tag in longest_list:
    ReportUtils.print(f"'{tag}'")

%timeit -r 1 -n 1 tags.most_words(5)

Теги с наибольшим количеством слов:
'Something for everyone in this one... saw it without and plan on seeing it with kids!': 16 слов
'the catholic church is the most corrupt organization in history': 10 слов
'villain nonexistent or not needed for good story': 8 слов
'06 Oscar Nominated Best Movie - Animation': 7 слов
'It was melodramatic and kind of dumb': 7 слов

Самые длинные теги:
'Something for everyone in this one... saw it without and plan on seeing it with kids!'
'the catholic church is the most corrupt organization in history'
'villain nonexistent or not needed for good story'
'r:disturbing violent content including rape'
'06 Oscar Nominated Best Movie - Animation'
1.26 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Развернутые теги демонстрируют переход от простых меток к полноценным высказываниям. Лидер - 16-словный тег, описывающий личный опыт просмотра с детьми. Социально-критический тег о католической церкви и аналитический о структуре сюжета показывают разнообразие пользовательских подходов.

Длинные теги выполняют функции мини-рецензий, сочетая личные впечатления, информационные пометки о наградах и возрастных ограничениях. Это свидетельствует о естественном развитии системы тегирования в сторону более содержательных описаний.

### 🎪 Теги с наибольшим количеством слов и самые длинные

In [48]:
intersection = tags.most_words_and_longest(5)
ReportUtils.print("Теги в топе по количеству слов и длине:")
intersection_list = ReportUtils.to_list(intersection)
for tag in intersection_list:
    ReportUtils.print(f"'{tag}'")

%timeit -r 1 -n 1 tags.most_words_and_longest(5)

Теги в топе по количеству слов и длине:
'06 Oscar Nominated Best Movie - Animation'
'villain nonexistent or not needed for good story'
'the catholic church is the most corrupt organization in history'
'Something for everyone in this one... saw it without and plan on seeing it with kids!'
1.74 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Теги, одновременно входящие в топ по длине и количеству слов, представляют особый интерес. Это развернутые высказывания, выполняющие множественные функции: анализ сюжета ("villain nonexistent"), социальная критика ("catholic church"), информационные пометки о наградах ("Oscar Nominated") и личный опыт просмотра ("saw it without kids").

Эти теги демонстрируют, что пользователи естественным образом развивают систему тегирования в сторону более содержательных и многофункциональных описаний, сочетая аналитический подход с личными впечатлениями.

### 🔍 Поиск тегов по ключевым словам

In [51]:
comedy_tags = tags.tags_with("comedy")
action_tags = tags.tags_with("action")
ReportUtils.print("Теги, содержащие 'comedy':")
comedy_list = ReportUtils.to_list(comedy_tags)
comedy_top = ReportUtils.slice_collection(comedy_list, 0, 8)
for tag in comedy_top:
    ReportUtils.print(f"{tag}")

ReportUtils.print("")
ReportUtils.print("Теги, содержащие 'action':")
action_list = ReportUtils.to_list(action_tags)
action_top = ReportUtils.slice_collection(action_list, 0, 8)
for tag in action_top:
    ReportUtils.print(f"{tag}")

%timeit -r 1 -n 1 tags.tags_with("comedy")

Теги, содержащие 'comedy':
Black comedy
Comedy
avant-garde romantic comedy
best comedy
black comedy
british comedy
comedy
dark comedy

Теги, содержащие 'action':
Action
action
action choreography
action packed
live action/animation
slow action
space action
532 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Наблюдается разнообразие комедийных поджанров: от классических "comedy" до специализированных "black comedy", "british comedy", "dark comedy". Пользователи четко различают оттенки юмора и национальные особенности комедий.

Включают как базовые обозначения "action", так и качественные характеристики ("action packed", "slow action") и технические аспекты ("action choreography"). Особый интерес представляет гибридный формат "live action/animation".

## 🔗 Анализ внешних ссылок и метрик

### 🎥 Мок-данные для анализа (IMDB)

In [55]:
movie_ids = [1, 2, 3]
imdb_data = links.get_imdb(movie_ids)
ReportUtils.print("Финансовые показатели первых трех фильмов:")
imdb_list = ReportUtils.to_list(imdb_data)
for data in imdb_list:
    movie_id, director, budget, gross, runtime = data
    ReportUtils.print(f"Фильм {movie_id}:")
    ReportUtils.print(f"Режиссер: {director}")
    ReportUtils.print(f"Бюджет: ${budget}")
    ReportUtils.print(f"Сборы: ${gross}")
    ReportUtils.print(f"Время: {runtime} мин")
    ReportUtils.print("")

%timeit -r 1 -n 1 links.get_imdb(movie_ids)

Финансовые показатели первых трех фильмов:
Фильм 3:
Режиссер: Howard Deutch
Бюджет: $$25,000,000.00
Сборы: $$71,518,503.00
Время: 6060 мин

Фильм 2:
Режиссер: Joe Johnston
Бюджет: $$65,000,000.00
Сборы: $$262,821,940.00
Время: 6240 мин

Фильм 1:
Режиссер: John Lasseter
Бюджет: $$30,000,000.00
Сборы: $$401,157,969.00
Время: 4860 мин

2.72 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Данные демонстрируют значительный разброс в бюджетах и кассовых сборах среди первых трех фильмов базы. Фильм 1 показывает наилучшее соотношение бюджета и сборов при минимальной продолжительности. Фильм 2, несмотря на самый высокий бюджет, демонстрирует уверенные сборы. Фильм 3 имеет скромные финансовые результаты при средней продолжительности.

Продолжительность фильмов не коррелирует напрямую с финансовым успехом, что подтверждает важность других факторов - режиссерского подхода, целевой аудитории и маркетинговой стратегии.

### 🎬 Топ режиссеров

In [58]:
top_dirs = links.top_directors(3)
ReportUtils.print("Топ-3 режиссера по количеству фильмов:")
directors_list = ReportUtils.to_list(top_dirs.items())

for director, count in directors_list:
    ReportUtils.print(f"{director}: {count} фильмов")

%timeit -r 1 -n 1 links.top_directors(3)

Топ-3 режиссера по количеству фильмов:
John Lasseter: 1 фильмов
Joe Johnston: 1 фильмов
Howard Deutch: 1 фильмов
9.6 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [59]:
expensive = links.most_expensive(5)
profitable = links.most_profitable(5)
cost_per_min = links.top_cost_per_minute(5)

ReportUtils.print("Топ-5 самых дорогих фильмов:")
expensive_list = ReportUtils.to_list(expensive.items())
for title, budget in expensive_list:
    ReportUtils.print(f"{title}: ${budget}")

ReportUtils.print("")
ReportUtils.print("Топ-5 самых прибыльных фильмов:")
profitable_list = ReportUtils.to_list(profitable.items())
for title, profit in profitable_list:
    ReportUtils.print(f"{title}: ${profit}")

ReportUtils.print("")
ReportUtils.print("Топ-5 по стоимости минуты экранного времени:")
cost_list = ReportUtils.to_list(cost_per_min.items())
for title, cost in cost_list:
    ReportUtils.print(f"{title}: ${cost:.2f}/мин")

%timeit -r 1 -n 1 links.most_expensive(5)

Топ-5 самых дорогих фильмов:
Jumanji (1995): $65000000.0
Toy Story (1995): $30000000.0

Топ-5 самых прибыльных фильмов:
Toy Story (1995): $343554033
Jumanji (1995): $197797249

Топ-5 по стоимости минуты экранного времени:
Jumanji (1995): $625000.00/мин
Toy Story (1995): $370370.37/мин
9.66 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [60]:
expensive = links.most_expensive(5)
profitable = links.most_profitable(5)
cost_per_min = links.top_cost_per_minute(5)

ReportUtils.print("Топ-5 самых дорогих фильмов:")
expensive_list = ReportUtils.to_list(expensive.items())
for title, budget in expensive_list:
    ReportUtils.print(f"{title}: ${budget}")

ReportUtils.print("")
ReportUtils.print("Топ-5 самых прибыльных фильмов:")
profitable_list = ReportUtils.to_list(profitable.items())
for title, profit in profitable_list:
    ReportUtils.print(f"{title}: ${profit}")

ReportUtils.print("")
ReportUtils.print("Топ-5 по стоимости минуты экранного времени:")
cost_list = ReportUtils.to_list(cost_per_min.items())
for title, cost in cost_list:
    ReportUtils.print(f"{title}: ${cost:.2f}/мин")

%timeit -r 1 -n 1 links.most_expensive(5)

Топ-5 самых дорогих фильмов:
Jumanji (1995): $65000000.0
Toy Story (1995): $30000000.0

Топ-5 самых прибыльных фильмов:
Toy Story (1995): $343554033
Jumanji (1995): $197797249

Топ-5 по стоимости минуты экранного времени:
Jumanji (1995): $625000.00/мин
Toy Story (1995): $370370.37/мин
9.67 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Данные выявляют двух явных лидеров - "Jumanji" (1995) и "Toy Story" (1995). "Jumanji" является самым дорогим проектом с бюджетом 65 млн долларов, однако "Toy Story" демонстрирует более высокую общую прибыль (343 млн долларов против 197 млн долларов).

Интересно, что при меньшем бюджете "Toy Story" показывает лучшую окупаемость, в то время как "Jumanji" имеет самую высокую стоимость минуты экранного времени ($625 тыс./мин). Это указывает на различную эффективность производства и маркетинговые стратегии двух студий..

### ⏱️ Анализ продолжительности фильмов

In [63]:
longest_movies = links.longest(3)
ReportUtils.print("Топ-3 по продолжительности:")
movies_list = ReportUtils.to_list(longest_movies.items())

for title, runtime in movies_list:
    ReportUtils.print(f"{title}: {runtime} минут")

%timeit -r 1 -n 1 links.longest(3)

Топ-3 по продолжительности:
Jumanji (1995): 104 минут
Grumpier Old Men (1995): 101 минут
Toy Story (1995): 81 минут
11.5 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Данные показывают трех лидеров по продолжительности с близкими показателями. "Джуманджи" (1995) - самый длинный фильм (104 минуты), незначительно опережающий "Взрослые пошлее некуда" (101 минута). "История игрушек" существенно короче (81 минута), что характерно для анимационного формата.

Разброс в продолжительности между лидерами составляет всего 23 минуты, что свидетельствует о стандартизации хронометража в mainstream-кинематографе середины 1990-х годов.

#  Итоги анализа киноданных

##  Ключевые открытия исследования

### О фильмах
**Год-рекордсмен**: 2002 год стал самым продуктивным по количеству выпущенных фильмов  
**Жанровые фавориты**: Drama, Comedy и Action — безоговорочные лидеры по популярности  
**Творческие эксперименты**: Отдельные картины объединяют до 8 жанров, создавая уникальные гибридные формы

### О зрительских оценках  
**Щедрость оценок**: Пользователи склонны к высоким оценкам (диапазон 3.5-4.5 баллов)  
**Феномены популярности**: Некоторые фильмы собирают в тысячи раз больше оценок, чем средние показатели  
**Поляризация мнений**: Отдельные работы вызывают диаметрально противоположные реакции

### О пользователях
**Элита платформы**: Небольшая группа энтузиастов формирует основу рейтинговой системы  
**Индивидуальные стили**: Каждый пользователь демонстрирует уникальный подход к оцениванию  
**Предсказуемость vs. спонтанность**: От консервативных до непредсказуемых паттернов поведения

### О тегах и метаданных
**Популярные категории**: comedy, action и superhero доминируют в пользовательской разметке  
**Разнообразие форматов**: От лаконичных ярлыков до развернутых описаний  
**Глубокая категоризация**: Теги создают альтернативную систему классификации контента

## Заключение

Данные MovieLens открывают уникальное окно в мир кинопредпочтений и пользовательского поведения. Наше исследование выявило системные закономерности, которые могут стать основой для:

- **Умных рекомендательных систем**, учитывающих не только жанры, но и паттерны оценок
- **Сегментации аудитории** на основе стилей просмотра и оценивания  
- **Прогнозирования успеха** фильмов на основе исторических данных

Кино остается искусством, но данные доказывают: даже в творчестве существуют свои законы и закономерности, которые помогают лучше понимать зрителей и создавать более качественный контент.